# Lecture 05: Transformer components

Ungraded classroom practice, in slide order. These small examples use Python's
standard library. No model, dataset, network connection, or GPU is needed.

- E01 follows the sinusoidal-frequency explanation.
- E02 follows the RoPE rotation explanation.
- E03 follows the multi-head mechanism.
- E04 follows residual addition, LayerNorm, and the order comparison.
- E05 follows the FFN mechanism and parameter-share explanation.

The running sentence is **Noa can be annoying but she is a great cat**.
The first six positions form the prefix through she. Numerical vectors are
invented for each mechanism, not states produced by a trained model.

All angles in the code are radians. Positions start at zero.
- E06: count an encoder with explicit parameter assumptions.

---

<!-- bilingual-zh -->

# 第 5 讲：Transformer 组件
按课件顺序进行不计分课堂练习；小型示例只用 Python 标准库，无需模型、数据集、网络或 GPU。E01 对应正弦频率；E02 对应 RoPE 旋转；E03 对应多头机制；E04 对应残差、LayerNorm 和顺序比较；E05 对应 FFN 及参数占比；E06 在明确假设下统计编码器参数。

贯穿句子为 **Noa can be annoying but she is a great cat**，前六个位置构成至 she 的前缀。每个机制的数值向量均为虚构，不是训练模型的状态。代码角度以弧度计，位置从 0 开始。


## Practice E01 · build one position vector · 2 minutes

For width 4, the pair frequencies are 1 and 0.01. Predict `PE(0)` on paper,
then add `PE(1)` to the toy embedding `e_can = [1, 0, 1, 0]` for can at position 1.

Here `e` is the already-scaled token embedding: the original Transformer uses
`sqrt(d_model) * E[token]` before adding the position vector.

Run the cells below to check your answer. Try position 2 and compare the two pairs.

---

<!-- bilingual-zh -->

## E01 · 构造一个位置向量 · 2 分钟
宽度为 4，两个坐标对的频率为 1 和 0.01。先手算 `PE(0)`，再将 `PE(1)` 加到位置 1 的 can 的玩具嵌入 `e_can=[1,0,1,0]`。这里的 `e` 已经过缩放：原始 Transformer 在加位置向量之前先计算 `sqrt(d_model) * E[token]`。运行下方单元核对，再试位置 2，比较两个坐标对。


In [ ]:
from math import sin, cos, sqrt, pi, isclose


def sinusoidal(position, width):
    if width <= 0 or width % 2:
        raise ValueError('Use a positive even width.')
    pairs = [(sin(position / 10000 ** (2 * i / width)),
              cos(position / 10000 ** (2 * i / width)))
             for i in range(width // 2)]
    return [value for pair in pairs for value in pair]


for position in (0, 1, 2):
    print(position, [round(value, 6) for value in sinusoidal(position, 4)])

In [ ]:
e = [1, 0, 1, 0]  # Invented, already-scaled embedding.
x = [content + position for content, position in zip(e, sinusoidal(1, 4))]
assert sinusoidal(0, 4) == [0, 1, 0, 1]
expected = [1.8414709848078965, 0.5403023058681398,
            1.0099998333341667, 0.9999500004166653]
assert all(isclose(a, b, abs_tol=1e-12) for a, b in zip(x, expected))
print('Input:', [round(value, 3) for value in x])

## Practice E02 · shift both positions · 2 minutes

Keep the supplied content vectors fixed at `q_she = k_Noa = (1, 0)` and use
the toy frequency 30 degrees per token. The running prefix puts she at
`m = 5` and Noa at `n = 0`. Predict their raw dot product, then shift both
positions by 3 to `(8, 3)` without changing the content vectors.

Next change their gap from 5 to 6. These are hypothetical position changes,
not a new sentence or a claim that a whole model is shift invariant. Compare
raw dot products before scaling, masking, and softmax. The toy frequency
is chosen for clear geometry; baseline RoPE starts at 1 radian per token.

---

<!-- bilingual-zh -->

## E02 · 同时平移两个位置 · 2 分钟
保持内容向量 `q_she=k_Noa=(1,0)` 不变，使用每词元旋转 30 度的玩具频率。前缀中 she 位于 `m=5`，Noa 位于 `n=0`；预测原始点积，再将两位置同时加 3 变为 `(8,3)`。然后将间隔从 5 改为 6。这是假设的位置变化，并非新句子或完整模型平移不变的断言。比较缩放、掩码和 softmax 之前的原始点积。玩具频率仅为便于观察几何关系；基础 RoPE 的首个频率为每词元 1 弧度。


In [ ]:
def rotate_pair(pair, angle):
    x, y = pair
    return [cos(angle) * x - sin(angle) * y,
            sin(angle) * x + cos(angle) * y]


def dot(left, right):
    return sum(a * b for a, b in zip(left, right))


theta = pi / 6
for m, n in [(5, 0), (8, 3), (6, 0), (9, 3)]:
    score = dot(rotate_pair([1, 0], m * theta), rotate_pair([1, 0], n * theta))
    expected = -(3 ** 0.5) / 2 if m - n == 5 else -1.0
    assert isclose(score, expected, abs_tol=1e-12)
    print(f'she at m={m}, Noa at n={n}, gap={m-n}: {score:.3f}')

### Check the full multi-pair identity

For column vectors, `R(mθ).T @ R(nθ) = R((n-m)θ)`: the signed offset is
**n minus m**. Repeat the check with different query and key vectors and a
frequency per coordinate pair. A shared shift preserves their dot product
and each rotation preserves vector length.

This identity holds for fixed content vectors. It does not assert that an
entire model produces the same outputs when its prompt content changes.

---

<!-- bilingual-zh -->

### 核对完整多坐标对恒等式
对列向量，`R(mθ).T @ R(nθ) = R((n-m)θ)`，有符号偏移为 **n 减 m**。换用不同查询和键向量，并对每对坐标使用不同频率，再次检查。共同平移保持点积，每次旋转保持向量长度。恒等式以内容向量固定为前提，并不声称整个模型在提示内容变化后输出不变。


In [ ]:
def rope(vector, position):
    width = len(vector)
    if width <= 0 or width % 2:
        raise ValueError('Use a positive even width.')
    pairs = [rotate_pair(vector[2*i:2*i+2], position / 10000 ** (2*i/width))
             for i in range(width // 2)]
    return [value for pair in pairs for value in pair]


q, k = [0.3, -0.8, 1.5, 0.2], [-0.2, 0.6, 0.9, -1.2]
m, n, shift = 5, 0, 3
score = dot(rope(q, m), rope(k, n))
assert isclose(dot(rope(q, m), rope(q, m)), dot(q, q), abs_tol=1e-12)
assert isclose(score, dot(q, rope(k, n-m)), abs_tol=1e-12)
assert isclose(score, dot(rope(q, m+shift), rope(k, n+shift)), abs_tol=1e-12)
print('Norm and relative-offset checks passed.')

### E02 follow-up · two coordinates as one complex number

A pair `(a, b)` is `z = a + ib`. Multiplication by `exp(i * angle)`
rotates it without changing its length. Predict the result for the supplied
she pair `(1, 0)` at position 5 with toy frequency π/6, then compare the two
implementations on non-axis-aligned pairs and both rotation directions.

This is the algebra behind the [complex-plane slide](index.html#/rope-complex-pairs).
It adds no new graded exercise or training run.

---

<!-- bilingual-zh -->

### E02 延伸 · 将两个坐标看作一个复数
坐标对 `(a,b)` 表示复数 `z=a+ib`。乘以 `exp(i*angle)` 会旋转而不改变长度。预测 she 的 `(1,0)` 在位置 5、玩具频率 π/6 下的结果；再用非轴对齐坐标对及两个旋转方向比较两种实现。这解释原文链接的复平面课件，不增加计分练习或训练任务。


In [ ]:
from cmath import exp

for pair in [(1.0, 0.0), (0.3, -0.8), (-1.2, 0.6)]:
    for angle in [0.0, 5 * pi / 6, -pi / 4]:
        z = complex(*pair)
        rotated = z * exp(1j * angle)
        real_pair = rotate_pair(pair, angle)
        assert isclose(rotated.real, real_pair[0], abs_tol=1e-12)
        assert isclose(rotated.imag, real_pair[1], abs_tol=1e-12)
        assert isclose(abs(rotated), abs(z), abs_tol=1e-12)

she_complex = complex(1, 0) * exp(1j * 5 * pi / 6)
assert isclose(she_complex.real, -sqrt(3) / 2)
assert isclose(she_complex.imag, 0.5)
print("she at 5:", (round(she_complex.real, 6), round(she_complex.imag, 6)))


### E02 follow-up · several rotation speeds inside one head

For head width 8 and base 10000, the four adjacent coordinate pairs use
frequencies `1, 0.1, 0.01, 0.001` radians per token. At she’s position 5,
the angles are `5, 0.5, 0.05, 0.005` radians. Noa at zero has no positional
rotation. These are the [four circles on the slide](index.html#/rope-pair-frequencies),
each with supplied content `(1, 0)` to isolate the change in frequency.

Next use **different, non-unit query and key pairs**. Sum the real parts of
`conj(q_pair) * k_pair` to obtain their real dot product. With rotations at
positions `m` and `n`, each pair contributes a phase `exp(i * (n - m) * theta)`.
Check this against the real-valued `rope` function and a shared shift of +3.
Keep content fixed: this checks the positional operation, not complete-model
invariance under prompt changes. Values V remain unchanged in standard RoPE.

---

<!-- bilingual-zh -->

### E02 延伸 · 一个头中的多种旋转速度
头宽为 8、base 为 10000 时，四个相邻坐标对的频率依次为每词元 `1,0.1,0.01,0.001` 弧度。she 在位置 5 时的角度为 `5,0.5,0.05,0.005`；Noa 在位置 0，不发生位置旋转。课件四个圆均使用内容 `(1,0)`，以单独观察频率变化。

再使用**不同且非单位长度的查询和键坐标对**。将 `conj(q_pair)*k_pair` 的实部求和得到实数点积；旋转位置为 m、n 时每对贡献相位 `exp(i*(n-m)*theta)`。与实值 `rope` 函数及同时平移 +3 比较。内容保持固定：这检查位置运算，不是完整模型对提示变化的不变性。标准 RoPE 不改变值向量 V。


In [ ]:
head_width = 8
pair_frequencies = [10000 ** (-2 * j / head_width) for j in range(head_width // 2)]
assert all(isclose(actual, expected) for actual, expected in
           zip(pair_frequencies, [1, 0.1, 0.01, 0.001]))
for j, frequency in enumerate(pair_frequencies):
    print(f"Pair {j}: theta={frequency:g}, she angle={5 * frequency:g} rad")

# Adjacent pairing: coordinates (0,1), (2,3), (4,5), (6,7).
def complex_rope(vector, position):
    width = len(vector)
    return [complex(vector[2 * j], vector[2 * j + 1])
            * exp(1j * position * 10000 ** (-2 * j / width))
            for j in range(width // 2)]

def complex_score(query_pairs, key_pairs):
    return sum((query.conjugate() * key).real
               for query, key in zip(query_pairs, key_pairs))

query = [0.3, -0.8, 1.5, 0.2, -0.6, 0.4, 0.7, -1.1]
key = [-0.2, 0.6, 0.9, -1.2, 0.5, 1.3, -0.4, 0.8]
m, n, common_shift = 5, 0, 3
rotated_query, rotated_key = complex_rope(query, m), complex_rope(key, n)
head_score = complex_score(rotated_query, rotated_key)
real_score = dot(rope(query, m), rope(key, n))
relative_score = complex_score(complex_rope(query, 0), complex_rope(key, n - m))
shifted_score = complex_score(complex_rope(query, m + common_shift),
                              complex_rope(key, n + common_shift))
assert isclose(head_score, real_score, abs_tol=1e-12)
assert isclose(head_score, relative_score, abs_tol=1e-12)
assert isclose(head_score, shifted_score, abs_tol=1e-12)
for original, rotated in [(query, rotated_query), (key, rotated_key)]:
    assert all(isclose(abs(before), abs(after), abs_tol=1e-12)
               for before, after in zip(complex_rope(original, 0), rotated))
print(f"Raw head score: {head_score:.6f}; shared shift: {shifted_score:.6f}")


## Practice E03 · multi-head dimensions · 1 minute

Use `d_model = 512`, `heads = 8`, and `tokens = 6` for Noa / can / be / annoying / but / she. Assume equal key and value
widths, each `d_model / heads`, as in the original Transformer's base model.

Before running the next cell, predict:

1. The width of one head.
2. The shape after concatenating all heads.
3. The shape of the output projection and final output.

The demonstration supplies invented head outputs so we can inspect the
concatenation axis. It does not simulate learned attention weights. Each
number labels a head and token row; no trained model or dataset is used.

---

<!-- bilingual-zh -->

## E03 · 多头维度 · 1 分钟
对 Noa / can / be / annoying / but / she 六个词元，使用 `d_model=512`、8 个头；假定键和值的每头宽度均为 `d_model/heads`，与原始 Transformer base 相同。运行前预测：每头宽度；拼接所有头后的形状；输出投影和最终输出的形状。演示用虚构头输出检查拼接轴，不模拟学到的注意力权重；数字用于标识头和词元行，不使用训练模型或数据集。


In [ ]:
n, d_model, h = 6, 512, 8
d_head = d_model // h
assert d_head == 64
# H[head][token][feature]: precomputed toy head outputs.
H = [[[100 * head + token] * d_head for token in range(n)] for head in range(h)]
# Join features for each token, keeping its row unchanged.
joined = [[value for head in H for value in head[token]] for token in range(n)]
assert (len(joined), len(joined[0])) == (6, 512)
for token in range(n):
    for head in range(h):
        assert joined[token][head * d_head:(head + 1) * d_head] == H[head][token]
projection_shape = (h * d_head, d_model)
output_shape = (n, projection_shape[1])
assert projection_shape == (512, 512) and output_shape == (6, 512)
print('One head:', (n, d_head))
print('Concatenated:', (len(joined), len(joined[0])))
print('W_O:', projection_shape, 'Output:', output_shape)

Each head has width **64**. The head outputs are **6 × 64**; concatenating their
features gives **6 × 512**. `W_O` is **512 × 512**, so the output remains **6 × 512**.
The number of token positions stays six.

Each head's query, key, and value projection is **512 × 64**. A head can use
all 512 input coordinates through its own learned projection; it is not
restricted to a fixed 64-coordinate slice of the raw input.

Reading: [Attention Is All You Need, section 3.2.2](https://arxiv.org/html/1706.03762v7#S3.S2).

---

<!-- bilingual-zh -->

**核对：**每头宽 **64**，每头输出为 **6×64**，沿特征维拼接得到 **6×512**；`W_O` 为 **512×512**，最终仍为 **6×512**，词元位置仍为 6。每头查询、键、值投影均为 **512×64**；每个头可通过自己的学习投影使用全部 512 个输入坐标，并非只能使用原输入中固定的 64 维切片。


## Practice E04 · normalize, then add the update · 2 minutes

For she, use the supplied toy state `x = [1, 3]`, `gamma = [1, 1]`, `beta = [0, 0]`, and
`eps = 1e-5`. The pre-LN update is given: `F(LN(x)) = [2, -1]`.

Before running the code, calculate the mean, population variance, normalized
vector, and residual output `y = x + F(LN(x))`. Can another token change
`LN(x)`? Distinguish the normalization from an attention update, which can
read allowed earlier tokens.

The variance divides by the feature count. Epsilon goes **inside** the square
root. Each token's feature vector has its own statistics. Learned scale and
bias are shared across token positions within one LayerNorm module.

---

<!-- bilingual-zh -->

## E04 · 先归一化，再加更新 · 2 分钟
对 she 使用玩具状态 `x=[1,3]`、`gamma=[1,1]`、`beta=[0,0]`、`eps=1e-5`；给定 pre-LN 更新 `F(LN(x))=[2,-1]`。手算均值、总体方差、归一化向量及残差输出 `y=x+F(LN(x))`。另一个词元能改变 `LN(x)` 吗？区分归一化与能读取允许的较早词元的注意力更新。

方差除以特征数，epsilon 在**平方根内部**。每个词元的特征向量有自己的统计量；同一 LayerNorm 模块的学习缩放和偏置跨词元位置共享。


In [ ]:
def layer_norm_features(vector, gamma, beta, eps=1e-5):
    """Normalize one token's features using population variance."""
    if not vector or not (len(vector) == len(gamma) == len(beta)):
        raise ValueError('Use matching nonempty feature vectors.')
    if eps <= 0:
        raise ValueError('Use a positive epsilon.')
    mean = sum(vector) / len(vector)
    variance = sum((value - mean) ** 2 for value in vector) / len(vector)
    return [scale * (value - mean) / sqrt(variance + eps) + bias
            for value, scale, bias in zip(vector, gamma, beta)]


x, gamma, beta = [1, 3], [1, 1], [0, 0]
mean = sum(x) / len(x)
variance = sum((value - mean) ** 2 for value in x) / len(x)
normalized = layer_norm_features(x, gamma, beta)
update = [2, -1]  # Given F(LN(x)); no model is trained here.
y = [value + change for value, change in zip(x, update)]
assert mean == 2 and variance == 1
expected = [-1 / sqrt(1.00001), 1 / sqrt(1.00001)]
assert all(isclose(a, b, abs_tol=1e-12) for a, b in zip(normalized, expected))
assert y == [3, 2]
print('Mean:', mean, 'Population variance:', variance)
print('LN(x):', [round(value, 6) for value in normalized], 'Residual output:', y)


### Check the axis and constant-vector case

The nested list below has shape `(B, T, d) = (1, 2, 2)`. Normalize its Noa and she toy
rows separately, then perturb only the second row. Predict which output row
can change. A constant input has zero centered features, so LayerNorm returns
`beta`; epsilon keeps the denominator nonzero.

---

<!-- bilingual-zh -->

### 检查归一化轴和常量向量
下方嵌套列表形状 `(B,T,d)=(1,2,2)`。分别归一化 Noa 和 she 的玩具行，再只扰动第二行，预测哪一输出行可能变化。常量输入的中心化特征为零，LayerNorm 返回 `beta`；epsilon 保证分母非零。


In [ ]:
def normalize_tokens(batch, gamma, beta):
    return [[layer_norm_features(token, gamma, beta) for token in sequence]
            for sequence in batch]


batch = [[[1, 3], [10, 14]]]  # Invented batch, token, and feature axes.
before = normalize_tokens(batch, gamma, beta)
batch[0][1] = [1000, -1000]  # Change only the other token.
after = normalize_tokens(batch, gamma, beta)
assert before[0][0] == after[0][0]
assert before[0][1] != after[0][1]
assert layer_norm_features([5, 5], [2, 3], [0.5, -0.5]) == [0.5, -0.5]
scaled = layer_norm_features(x, [2, 3], [0.5, -0.5])
assert all(isclose(a, b, abs_tol=1e-12) for a, b in zip(
    scaled, [2 * expected[0] + 0.5, 3 * expected[1] - 0.5]))
print('Token independence, constant-input, and learned-affine checks passed.')


The normalized vector is approximately **(-0.999995, 0.999995)** and the
pre-LN output is **(3, 2)**. Add the update to the original `x`.

Changing the other token leaves this token's LayerNorm unchanged. This check
isolates normalization; attention can still mix information from allowed
positions. LayerNorm recomputes statistics from each input in both training
and evaluation, and uses no batch running averages.

For a zero update, a pre-LN sublayer passes `x` through unchanged; a post-LN
sublayer would return `LN(x)`. After learned scale and bias, a LayerNorm output
need not have mean zero or variance one.

Readings: [Ba et al., §3](https://arxiv.org/html/1607.06450v1#S3),
[PyTorch LayerNorm formula](https://github.com/pytorch/pytorch/blob/v2.8.0/torch/nn/modules/normalization.py#L88),
and [Xiong et al., §§2–4](https://proceedings.mlr.press/v119/xiong20b.html).

---

<!-- bilingual-zh -->

**核对：**归一化向量约为 **(-0.999995,0.999995)**，pre-LN 输出为 **(3,2)**；将更新加到原始 `x`。改变另一词元不会改变此词元的 LayerNorm；这里只隔离归一化，注意力仍可混合允许位置的信息。LayerNorm 在训练与评估时都根据当前输入重新计算统计量，不使用批次运行平均值。

更新为零时，pre-LN 子层原样传递 `x`，post-LN 则返回 `LN(x)`。经过可学习缩放与偏置后，LayerNorm 输出不一定均值为零、方差为一。参考资料见英文链接。


## Practice E05 · apply the FFN to one token · 2 minutes

For she, use the invented row vector `x = [1, -2]`, zero biases, and these matrices:

```text
W1 = [[1, -1,  2],      W2 = [[1,  0],
      [0,  1, -1]]            [0,  1],
                              [1, -1]]
```

Compute `x W1`, apply ReLU coordinate by coordinate, and multiply by `W2`.
Can changing another token's supplied input change this token's FFN output?
The widths here are **2 → 3 → 2**. The diagram's 4× expansion is a design choice.

---

<!-- bilingual-zh -->

## E05 · 将 FFN 应用于一个词元 · 2 分钟
对 she 使用虚构行向量 `x=[1,-2]`、零偏置及上方两个矩阵。计算 `x W1`，逐坐标应用 ReLU，再乘 `W2`。改变另一个词元给定的输入，能否改变这个词元的 FFN 输出？此处宽度为 **2→3→2**；图中的四倍扩展是一种设计选择。


In [ ]:
def affine_features(vector, weights, bias):
    """Multiply a row vector by a weight matrix and add the bias."""
    return [sum(vector[i] * weights[i][j] for i in range(len(vector))) + bias[j]
            for j in range(len(bias))]


ffn_w1 = [[1, -1, 2], [0, 1, -1]]
ffn_w2 = [[1, 0], [0, 1], [1, -1]]
ffn_b1, ffn_b2 = [0, 0, 0], [0, 0]


def token_ffn(vector):
    expanded = affine_features(vector, ffn_w1, ffn_b1)
    activated = [max(0, value) for value in expanded]
    return affine_features(activated, ffn_w2, ffn_b2)


ffn_x = [1, -2]
expanded = affine_features(ffn_x, ffn_w1, ffn_b1)
activated = [max(0, value) for value in expanded]
ffn_output = token_ffn(ffn_x)
assert expanded == [1, -3, 4]
assert activated == [1, 0, 4]
assert ffn_output == [5, -4]
print('Expanded:', expanded, 'After ReLU:', activated, 'FFN output:', ffn_output)


### Token independence and nonlinearity

Apply the same FFN to the supplied Noa and she toy vectors. Change only the second input and
check which output changes. This isolates the FFN; attention earlier in the
block can still change the contextual vector supplied to it.

Also compare `FFN(x) + FFN(-x)` with `FFN(0)`. With zero biases, a linear map
would make these equal. Predict what ReLU changes.

---

<!-- bilingual-zh -->

### 词元独立性与非线性
将同一个 FFN 分别应用于提供的 Noa 和 she 玩具向量，只改变第二个输入，检查哪一个输出变化。这里单独考察 FFN；模块前面的注意力仍可能改变输入给 FFN 的上下文向量。再比较 `FFN(x)+FFN(-x)` 与 `FFN(0)`：零偏置线性映射会使两者相等。预测 ReLU 会带来什么变化。


In [ ]:
ffn_inputs = [[1, -2], [0, 1]]
outputs_before = [token_ffn(token) for token in ffn_inputs]
changed_inputs = [ffn_inputs[0], [1000, -1000]]
outputs_after = [token_ffn(token) for token in changed_inputs]
assert outputs_before == [[5, -4], [0, 1]]
assert outputs_before[0] == outputs_after[0]
assert outputs_before[1] != outputs_after[1]
assert (len(outputs_before), len(outputs_before[0])) == (2, 2)

opposite_output = token_ffn([-value for value in ffn_x])
assert opposite_output == [0, 3]
assert [a + b for a, b in zip(ffn_output, opposite_output)] != token_ffn([0, 0])
print('Shared parameters, preserved token rows, and nonlinearity checks passed.')


### Check the two-thirds parameter share

For standard multi-head attention, the aggregate Q, K, V, and output matrices
contain `4 * d**2` weights. The two FFN matrices contain `2 * d * d_ff` weights.
At `d_ff = 4*d`, the FFN has two thirds of these **block matrix parameters**.
This count excludes embeddings, biases, normalization, and any extra attention
sublayers. Parameter count is one description of capacity, not a performance score.

---

<!-- bilingual-zh -->

### 核对三分之二参数占比
标准多头注意力的 Q、K、V 和输出矩阵总计 `4*d**2` 个权重；两个 FFN 矩阵共有 `2*d*d_ff` 个权重。$d_{ff}=4d$ 时，FFN 占这些**块内矩阵参数**的三分之二。此统计不含嵌入、偏置、归一化和额外注意力子层。参数量只是容量的一种描述，不是性能分数。


In [ ]:
d, d_ff = 512, 2048
attention_parameters = 4 * d * d
ffn_parameters = 2 * d * d_ff
ffn_share = ffn_parameters / (attention_parameters + ffn_parameters)
assert attention_parameters == 1048576
assert ffn_parameters == 2097152
assert isclose(ffn_share, 2 / 3)
# A narrower hidden layer changes the share even before adding embeddings.
narrow_ffn = 2 * d * (2 * d)
assert narrow_ffn / (attention_parameters + narrow_ffn) == 0.5
print('Attention matrix parameters:', attention_parameters)
print('FFN matrix parameters:', ffn_parameters, 'Block share:', round(ffn_share, 4))


The intermediate vectors are **(1, -3, 4)** and **(1, 0, 4)**. The FFN output
is **(5, -4)**, before residual addition. All token positions use the same weights,
but each supplied vector is processed separately.

The nonlinear activation matters: `FFN(x) + FFN(-x) = [5, -1]`, whereas
`FFN([0, 0]) = [0, 0]`. Attention also has a nonlinear softmax, so this example
does not imply that removing the FFN makes an entire Transformer linear.

At widths 512 and 2048, the FFN matrices have 2,097,152 parameters and the
attention matrices have 1,048,576. The FFN share is 2/3 of their total.
Whole-model accounting must also include the other components and architecture.

Readings: [Vaswani et al., §3.3](https://arxiv.org/html/1706.03762v7#S3.S3)
and [GPT-2's GELU MLP](https://github.com/openai/gpt-2/blob/master/src/model.py).

---

<!-- bilingual-zh -->

**核对：**中间向量为 **(1,-3,4)** 和 **(1,0,4)**，残差相加前的 FFN 输出为 **(5,-4)**。所有位置共用权重，但每个输入向量分别处理。非线性激活很重要：`FFN(x)+FFN(-x)=[5,-1]`，而 `FFN([0,0])=[0,0]`。注意力也有非线性 softmax，因此这不能说明移除 FFN 后整个 Transformer 就变成线性。

宽度 512 和 2048 时，FFN 矩阵共有 2,097,152 个参数，注意力矩阵共有 1,048,576 个，FFN 占两者总数的 2/3。全模型统计还须包括其他组件与具体架构。


## Practice E06 · count an encoder · 3 minutes

Use 12 layers, model width 768, 12 heads, vocabulary size 30,522, and FFN width `4 * width`.
Count token embeddings, all linear weights and biases, and two affine LayerNorms per layer.
Use sinusoidal positions. Exclude segment embeddings, embedding normalization, pooling,
extra final normalization, and prediction heads. This is the instructor's encoder-only
exercise, not the paper's base/big model or an exact BERT parameter count.

Predict the component counts before running the explicit shape tally below.
Splitting a fixed model width into more heads does not create another full-width matrix per head.

---

<!-- bilingual-zh -->

## E06 · 统计编码器参数 · 3 分钟
使用 12 层、模型宽度 768、12 个头、词表大小 30,522、FFN 宽度为模型宽度的 4 倍。统计词元嵌入、所有线性权重与偏置，以及每层两个带仿射参数的 LayerNorm；位置使用固定正弦编码。不计段嵌入、嵌入归一化、池化、额外最终归一化和预测头。这是教师的仅编码器练习，并非论文 base/big 模型或精确 BERT 参数量。运行显式形状统计前先预测各组件；固定模型宽度拆成更多头不会为每个头新增一套完整宽度矩阵。


In [ ]:
from math import prod


def encoder_parameter_counts(layers, width, heads, vocabulary):
    """Count the specified encoder, including linear biases and LayerNorm."""
    if any(type(value) is not int or value <= 0 for value in (layers, width, heads, vocabulary)):
        raise ValueError("dimensions and layer count must be positive integers")
    if width % heads:
        raise ValueError("model width must be divisible by heads")
    head_width = width // heads
    shapes = {
        "query.weight": (heads * head_width, width), "query.bias": (heads * head_width,),
        "key.weight": (heads * head_width, width), "key.bias": (heads * head_width,),
        "value.weight": (heads * head_width, width), "value.bias": (heads * head_width,),
        "output.weight": (width, heads * head_width), "output.bias": (width,),
        "ffn_expand.weight": (4 * width, width), "ffn_expand.bias": (4 * width,),
        "ffn_contract.weight": (width, 4 * width), "ffn_contract.bias": (width,),
        "norm1.scale": (width,), "norm1.shift": (width,),
        "norm2.scale": (width,), "norm2.shift": (width,),
    }
    one_layer = sum(prod(shape) for shape in shapes.values())
    embeddings = prod((vocabulary, width))
    return {"embeddings": embeddings, "one_layer": one_layer,
            "stack": layers * one_layer, "total": embeddings + layers * one_layer}


parameter_counts = encoder_parameter_counts(12, 768, 12, 30522)
assert parameter_counts == {"embeddings": 23_440_896, "one_layer": 7_087_872,
                            "stack": 85_054_464, "total": 108_495_360}
assert parameter_counts["total"] == 30522 * 768 + 12 * (12 * 768**2 + 13 * 768)
assert encoder_parameter_counts(12, 768, 1, 30522) == parameter_counts
print(parameter_counts)


**Check:** embeddings contain **23,440,896** parameters; one layer contains **7,087,872**;
the 12-layer stack contains **85,054,464**. The total is **108,495,360**, about **108.50M**.

For one layer: attention has `4*d*d + 4*d`, the FFN has `8*d*d + 5*d`,
and two LayerNorms have `4*d`. Add `V*d` once for the token table.
Residual additions, dropout, softmax, and fixed sinusoidal positions have no learned parameters.
Different choices about positional embeddings or output heads produce different totals.

---

<!-- bilingual-zh -->

**核对：**嵌入 **23,440,896** 个参数；每层 **7,087,872**；12 层共 **85,054,464**；总计 **108,495,360**，约 **108.50M**。每层注意力为 `4*d*d+4*d`，FFN 为 `8*d*d+5*d`，两个 LayerNorm 为 `4*d`；词元表 `V*d` 只加一次。残差相加、dropout、softmax 和固定正弦位置没有学习参数。位置嵌入或输出头选择不同，总量也不同。


## Read a paper's training-cost comparison

[Attention Is All You Need, Table 2](https://arxiv.org/html/1706.03762v7#S6.S1)
reports estimated training FLOPs. For English–French, compare Transformer big
(`2.3e19`) with single ConvS2S (`1.5e20`) and its ensemble (`1.2e21`).
Calculate both ratios. These are neither measured inference-speed ratios nor
controlled wall-clock comparisons on identical hardware.

---

<!-- bilingual-zh -->

## 阅读论文的训练成本比较
原文链接的《Attention Is All You Need》表 2 报告估算训练 FLOPs。对于英法翻译，比较 Transformer big（`2.3e19`）与单个 ConvS2S（`1.5e20`）及其集成（`1.2e21`），计算两个比值。这些不是实测推理速度比，也不是相同硬件上的受控墙钟时间比较。


In [ ]:
single_cost_ratio = 1.5e20 / 2.3e19
ensemble_cost_ratio = 1.2e21 / 2.3e19
assert isclose(single_cost_ratio, 6.521739130434782)
assert isclose(ensemble_cost_ratio, 52.17391304347826)
print(f"Single ConvS2S / Transformer big: {single_cost_ratio:.2f}")
print(f"ConvS2S ensemble / Transformer big: {ensemble_cost_ratio:.2f}")


## Readings

- [Attention Is All You Need, sections 3.4–3.5](https://arxiv.org/html/1706.03762v7#S3.S5).
- [RoFormer, sections 3.2.1–3.2.2](https://arxiv.org/html/2104.09864v5).

The notebook and diagrams use original, invented numerical examples.
- [Attention Is All You Need, sections 5–6](https://arxiv.org/html/1706.03762v7#S5): experimental setup, translation, ablations, and parsing.
- [Reformer (2020)](https://arxiv.org/abs/2001.04451) and [Linformer (2020)](https://arxiv.org/abs/2006.04768): later efficiency methods.
- [Narang et al. (2021)](https://aclanthology.org/2021.emnlp-main.465/): transfer of architecture modifications.

- [EleutherAI: Rotary Embeddings: A Relative Revolution (2021)](https://blog.eleuther.ai/rotary-embeddings/): read Intuition, Derivation, then Implementation alongside the E02 follow-ups.

---

<!-- bilingual-zh -->

## 阅读资料
原文链接涵盖 Transformer 的位置编码、实验与消融，RoFormer 的旋转位置编码，后续高效方法 Reformer、Linformer，架构修改的迁移研究，以及 EleutherAI 的旋转嵌入直觉、推导和实现。Notebook 与图示使用原创虚构数值示例。
